# Spleeter source separation

Edit the variables in the config cell to try different models, inputs, and output filenames.

If your audio file is not already WAV, `soundfile` may still read it depending on your local codec support.

Installation appears to be simply
```bash
pip install sherpa-onnx
```

However `soundfile`, `numpy` may be required

In [4]:
import time
from pathlib import Path

import numpy as np
import sherpa_onnx
import soundfile as sf

In [5]:
def create_offline_source_separation(vocals_model: Path, accompaniment_model: Path, num_threads: int = 1, provider: str = "cpu"):
    if not vocals_model.is_file():
        raise ValueError(f"{vocals_model} does not exist.")

    if not accompaniment_model.is_file():
        raise ValueError(f"{accompaniment_model} does not exist.")

    config = sherpa_onnx.OfflineSourceSeparationConfig(
        model=sherpa_onnx.OfflineSourceSeparationModelConfig(
            spleeter=sherpa_onnx.OfflineSourceSeparationSpleeterModelConfig(
                vocals=str(vocals_model),
                accompaniment=str(accompaniment_model),
            ),
            num_threads=num_threads,
            debug=False,
            provider=provider,
        )
    )
    if not config.validate():
        raise ValueError("Please check your config.")

    return sherpa_onnx.OfflineSourceSeparation(config)


def load_audio(wav_file: Path):
    if not wav_file.is_file():
        raise ValueError(f"{wav_file} does not exist")

    samples, sample_rate = sf.read(str(wav_file), dtype="float32", always_2d=True)
    samples = np.transpose(samples)
    assert samples.shape[1] > samples.shape[0], (
        f"You should use (num_channels, num_samples). {samples.shape}"
    )
    assert samples.dtype == np.float32, f"Expect np.float32 as dtype. Given: {samples.dtype}"

    return samples, sample_rate


def run_separation(vocals_model: Path, accompaniment_model: Path, input_audio: Path, output_vocals: Path, output_non_vocals: Path, num_threads: int = 1, provider: str = "cpu"):
    _start = time.perf_counter()
    sp = create_offline_source_separation(
        vocals_model=vocals_model,
        accompaniment_model=accompaniment_model,
        num_threads=num_threads,
        provider=provider,
    )
    _after_offline_source_separation = time.perf_counter()
    print(f"create_offline_source_separation() takes {_after_offline_source_separation - _start:.3f} seconds")
    samples, sample_rate = load_audio(input_audio)
    samples = np.ascontiguousarray(samples)
    _after_load_audio = time.perf_counter()
    print(f"load_audio() takes {_after_load_audio - _after_offline_source_separation:.3f} seconds")

    start = time.perf_counter()
    output = sp.process(sample_rate=sample_rate, samples=samples)
    end = time.perf_counter()

    print("output.sample_rate", output.sample_rate)
    assert len(output.stems) == 2, len(output.stems)

    vocals = np.transpose(output.stems[0].data)
    non_vocals = np.transpose(output.stems[1].data)

    sf.write(str(output_vocals), vocals, samplerate=output.sample_rate)
    sf.write(str(output_non_vocals), non_vocals, samplerate=output.sample_rate)
    _after_write_audio = time.perf_counter()
    print(f"write_audio() takes {_after_write_audio - end:.3f} seconds")
    elapsed_seconds = end - start
    audio_duration = samples.shape[1] / sample_rate
    real_time_factor = elapsed_seconds / audio_duration

    print(f"Saved to {output_vocals} and {output_non_vocals}")
    print(f"Elapsed seconds: {elapsed_seconds:.3f}")
    print(f"Audio duration in seconds: {audio_duration:.3f}")
    print(f"RTF: {elapsed_seconds:.3f}/{audio_duration:.3f} = {real_time_factor:.3f}")

    return output

In [9]:
# Experiment settings
VOCALS_MODEL = Path("./sherpa/sherpa-onnx-spleeter-2stems-fp16/vocals.fp16.onnx")
ACCOMPANIMENT_MODEL = Path("./sherpa/sherpa-onnx-spleeter-2stems-fp16/accompaniment.fp16.onnx")
INPUT_AUDIO = Path("./sherpa/opalite.mp3")

OUTPUT_VOCALS = Path("./sherpa/spleeter-vocals.wav")
OUTPUT_NON_VOCALS = Path("./sherpa/spleeter-non-vocals.wav")

# Increase this if you want to compare CPU parallelism.
NUM_THREADS = 8
PROVIDER = "cpu"

# Run with the editable values above.
run_separation(
    vocals_model=VOCALS_MODEL,
    accompaniment_model=ACCOMPANIMENT_MODEL,
    input_audio=INPUT_AUDIO,
    output_vocals=OUTPUT_VOCALS,
    output_non_vocals=OUTPUT_NON_VOCALS,
    num_threads=NUM_THREADS,
    provider=PROVIDER,
)

create_offline_source_separation() takes 0.179 seconds
load_audio() takes 0.276 seconds
output.sample_rate 44100
write_audio() takes 0.394 seconds
Saved to sherpa\spleeter-vocals.wav and sherpa\spleeter-non-vocals.wav
Elapsed seconds: 6.833
Audio duration in seconds: 233.758
RTF: 6.833/233.758 = 0.029


create_offline_source_separation() is negligible

load_audio() too, also accept .mp3

Typical time real-time factor 0.032 (for i7 12700K with 8 threads)

Even this is significantly faster than UVR-MDX, which is already 0.226 with i7 12700K 20 thread maxxed out.

### Configuration and Code
```
NUM_THREADS = 8
```
Offical Source: https://github.com/k2-fsa/sherpa-onnx/blob/master/python-api-examples/offline-source-separation-spleeter.py

Specify the number of threads to use for processing, can be adjusted, or use `os.cpu_count()`, the best spot for i7 12700K seems to be 8 threads.

Unlike Demucs, where calling the code without models will download the models, sherpa_onnx + Spleeter requires the models to be downloaded beforehand. We can expose this as an environment variable.

Download location according to the code.
```
wget https://github.com/k2-fsa/sherpa-onnx/releases/download/source-separation-models/sherpa-onnx-spleeter-2stems-fp16.tar.bz2
```

Other models: https://github.com/k2-fsa/sherpa-onnx/releases/tag/source-separation-models

This file also must be unzipped, in the file, there are two files, `accompaniment.onnx` and `vocals.fp16.onnx` both are needed, there are also these models which may provide different speed and accuracy.

```
sherpa-onnx-spleeter-2stems-fp16.tar.bz2
33.6 MB May 23, 2025
sherpa-onnx-spleeter-2stems-int8.tar.bz2
46.4 MB May 23, 2025
sherpa-onnx-spleeter-2stems.tar.bz2
71.2 MB May 23, 2025
```

Although sherpa_onnx recommend using .WAV files are input, it seems to work with .mp3 files as well, but the output will be .WAV files. To output a customizaed MP3 with different bitrate, the library does not provide switches, but we can use `ffmpeg` to convert the output WAV files to MP3 with a specified bitrate.

```powershell
ffmpeg -i output_vocals.wav -b:a 192k spleeter_vocals.mp3
```

GPU acceleration may be possible, but this notebook only explores CPU usage. Even if the main app requests GPU, we should still make it use CPU, we need to handle sherpa differently than Demucs. Since the speed on CPU is already acceptable, this library would be used as a replacement for Demucs on low-end systems.

The quality of the output is not as good as Demucs, in addition the bitrate is equivalent to 64/96 kbps, this is a sacrifice for no-GPU systems. So the user could pass in a higher bitrate which the API would support (for compatibility and standardization with main app settings, so we can use a single MP3 and bitrate value for both Demucs and Spleeter), but the output quality won't increase.

#### Warning

Sherpa-onnx and Spleeter do not support mp4 or even .aac audio files, only supported are mp3 and wav. So if a mp4 is passed into, the app need to check whether audio codec in the video is mp3, if so, use -c:a copy to extract it otherwise re-encode it to mp3 or wav, the run inference on the mp3/wav file.